# 03 · Evaluation

Uses the trained model saved by `01_entity_model.ipynb` (`entity_identifier.pt`).
Run All.

| Section | What it does |
|---|---|
| 1. Training curve | dev F1 per epoch, as a figure for the report |
| 2. Threshold tuning | picks the best confidence threshold on the **dev** set |
| 3. Test scores | the numbers for the report |
| 4. Reworded labels | does the model still work with *people* / *company* / *place*? |
| 5. Demo | your own text and labels |
| 6. For teammates | span vectors for coreference and relation extraction |

Model, dataset and threshold settings come from notebook 01's settings cell.

In [ ]:
# Load the model code from 01_entity_model.ipynb (its cells tagged "definitions"):
# settings, helpers, dataset loaders, the model and the training functions.
# Nothing is trained here. Keep this notebook in the same folder as notebook 01.
import json as _json
from pathlib import Path as _Path

_nb = _json.loads(_Path("01_entity_model.ipynb").read_text(encoding="utf-8"))
for _cell in _nb["cells"]:
    if _cell["cell_type"] == "code" and "definitions" in _cell["metadata"].get("tags", []):
        exec("".join(_cell["source"]), globals())
print("Loaded the model code from 01_entity_model.ipynb")


# Evaluation settings
THRESHOLD = CONFIG["THRESHOLD"]                 # set to the best value from section 2
THRESHOLDS_TO_TRY = [0.3, 0.4, 0.5, 0.6, 0.7]
RENAMED_LABELS = {"person": "people", "organization": "company", "location": "place"}

model = load_trained()
dev_sets = load_split("dev")
print(f"Loaded {CHECKPOINT_PATH.name}; dev sets: " + ", ".join(f"{k}={len(v)}" for k, v in dev_sets.items()))

## 1. Training curve

Dev F1 after each epoch (epoch 0 = before training).

In [ ]:
log_path = LOG_PATH if LOG_PATH.exists() else HERE / "training_log.json"   # older scripts' name
if log_path.exists():
    import matplotlib.pyplot as plt
    log = json.loads(log_path.read_text())
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for name in log[-1]["dev"]:
        ax.plot([e["epoch"] for e in log], [e["dev"][name]["f1"] for e in log], marker="o", label=name)
    ax.set_xlabel("epoch"); ax.set_ylabel("dev entity F1"); ax.set_ylim(0, 100)
    ax.grid(alpha=0.3); ax.legend(); ax.set_title("Entity identification: dev F1 per epoch")
    plt.show()
else:
    print("No training log found (the model was downloaded rather than trained here).")

## 2. Threshold tuning (dev set)

A span is kept when its best label scores above the threshold. The model runs once
on the dev set; each threshold is then scored. Put the best value into `THRESHOLD`
(in the first cell) and into notebook 01's settings. Never tune on the test set.

In [ ]:
@torch.no_grad()
def predict_probs(model, docs):
    outputs = []
    for batch in batches(docs, CONFIG["BATCH_SIZE"]):
        for res in model.forward(batch, compute_loss=False)["docs"]:
            outputs.append((res["doc"], res["spans"], torch.sigmoid(res["logits"]).cpu().tolist()))
    return outputs


def score_at(outputs, threshold):
    preds = [decode_mentions(spans, probs, doc.entity_labels, doc, threshold)
             for doc, spans, probs in outputs]
    s = entity_prf(preds, [doc.gold_mentions for doc, _, _ in outputs])
    s["n_pred"] = sum(map(len, preds))
    s["n_gold"] = sum(len(doc.gold_mentions) for doc, _, _ in outputs)
    return s


for name, docs in dev_sets.items():
    outputs = predict_probs(model, docs)
    results = {t: score_at(outputs, t) for t in THRESHOLDS_TO_TRY}
    print(f"{name}:  threshold      P       R      F1   predicted / gold")
    for t, s in results.items():
        print(f"           {t:5.2f}  {s['precision']:6.2f}  {s['recall']:6.2f}  {s['f1']:6.2f}   "
              f"{s['n_pred']} / {s['n_gold']}")
    best = max(results, key=lambda t: results[t]["f1"])
    print(f"  -> best threshold on dev: {best}   (currently using {THRESHOLD})")

## 3. Test scores (for the report)

Uses `THRESHOLD`. Decide settings on the dev set only; the test score is the final,
unbiased measurement.

In [ ]:
test_sets = load_split("test")
print(f"threshold {THRESHOLD}")
print(f"{'dataset':10} {'P':>7} {'R':>7} {'F1':>7}   predicted / gold")
for name, docs in test_sets.items():
    s, _ = evaluate(model, docs, threshold=THRESHOLD)
    print(f"{name:10} {s['precision']:7.2f} {s['recall']:7.2f} {s['f1']:7.2f}   {s['n_pred']} / {s['n_gold']}")

## 4. Reworded labels (dev set)

Swaps label names (e.g. *person* → *people*) in both the label set and the gold
answers. If F1 stays close to the original, the model understands what the labels
**mean**, which supports the open-label design.

In [ ]:
def renamed(docs, mapping):
    out = []
    for doc in docs:
        new = Document(**doc.__dict__)
        new.entity_labels = [mapping.get(l, l) for l in doc.entity_labels]
        new.gold_mentions = [{**m, "label": mapping.get(m["label"], m["label"])}
                             for m in doc.gold_mentions]
        out.append(new)
    return out


print("renamed:", RENAMED_LABELS)
for name, docs in dev_sets.items():
    original, _ = evaluate(model, docs, threshold=THRESHOLD)
    reworded, _ = evaluate(model, renamed(docs, RENAMED_LABELS), threshold=THRESHOLD)
    print(f"{name:9} original labels F1 {original['f1']:6.2f}   reworded labels F1 {reworded['f1']:6.2f}")

## 5. Demo: your own text

Edit `TEXT` and `LABELS`; labels can be any words.

In [ ]:
TEXT = "Maya founded Northstar Labs in Berlin in 2019. The company makes biosensors."
LABELS = ["person", "organization", "location", "time"]

result = model.predict([document_from_text(TEXT, LABELS)], threshold=THRESHOLD)[0]
for m in result["mentions"]:
    print(f"  {m['text']!r:30} -> {m['label']:15} ({m['score']:.2f})")
if not result["mentions"]:
    print("  (nothing above the threshold; try a lower THRESHOLD)")

## 6. For teammates: coreference and relation extraction

- **At run time**, `predict` returns the kept mentions and one 2304-number vector per
  mention (`mention_vectors`) for the coreference module.
- **During training**, coreference should use the gold mentions instead:
  `model.represent_spans(word_vectors, gold_spans)` works for spans of any width.
- **Relation labels** can reuse the shared BERT label encoder:
  `model.encode_label_text([...])` (apply your own 768 → 768 projection).
- **Joint training:** add the coreference and relation losses to `model(docs)["loss"]`.

In [ ]:
doc = document_from_text(TEXT, LABELS)
example = model.predict([doc])[0]
print("kept mentions  :", [m["text"] for m in example["mentions"]])
print("mention vectors:", tuple(example["mention_vectors"].shape))

with torch.no_grad():
    out = model([doc], compute_loss=False)["docs"][0]
    gold_spans = [(0, 0), (2, 3)]                 # 'Maya', 'Northstar Labs'
    print("gold span vectors     :", tuple(model.represent_spans(out["word_vectors"], gold_spans).shape))
    print("relation label vectors:", tuple(model.encode_label_text(["founded by", "located in"]).shape))